# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(tidyverse)
library(WGCNA)
library(magrittr)
library(biomaRt)
library(dplyr)

# Load Metadata

In [6]:
# Load WGCNA metadata
metadata <- read.csv("metadata.csv")

# Set row names
rownames(metadata) <- metadata$Sample_Name
metadata$Sample_Name <- NULL

# Clean up rows from DEG analysis
metadata$Gene <- NULL
metadata$Sample_ID <- NULL
metadata$Cell_Type <- NULL
metadata$Genotype <- NULL
metadata$Time_Point <- NULL
metadata$Group <- NULL
metadata$Replicate <- NULL
metadata$Clean_Sample_Name <- NULL
metadata$Batch <- NULL

# Remove comparisons we are not interested in 
metadata$D18_WT_vs_D46_WT <- NULL
metadata$D46_WT_vs_D90_WT <- NULL
metadata$D18_WT_vs_D90_WT <- NULL
metadata$D18_MUT_vs_D46_MUT <- NULL
metadata$D46_MUT_vs_D90_MUT <- NULL
metadata$D18_MUT_vs_D90_MUT <- NULL


# Sort metadata by row names
metadata <- metadata[order(rownames(metadata)), ]

# View
head(metadata)

,D18_MUT_vs_D18_WT,D46_MUT_vs_D46_WT,D90_MUT_vs_D90_WT,Decreased_GAD1_Expression,Decreased_FOXG1_Expression,Smaller_Organoid_Area,Impaired_Neurite_Outgrowth
,<int>,<int>,<int>,<int>,<int>,<int>,<int>
CO_R270X_B1_d18_A1,1,NA,NA,NA,1,NA,NA
CO_R270X_B1_d18_A2,1,NA,NA,NA,1,NA,NA
CO_R270X_B1_d18_A3,1,NA,NA,NA,1,NA,NA
CO_R270X_B1_d18_D1,0,NA,NA,NA,0,NA,NA
CO_R270X_B1_d18_D2,0,NA,NA,NA,0,NA,NA
CO_R270X_B1_d18_D3,0,NA,NA,NA,0,NA,NA


# Load Data

In [7]:
# Read in data
MEs0 <- read.table("08_WGCNA/MECP2_module_eigengenes.txt", header = TRUE, sep = "\t")

# Covnert to df
MEs0 <- as.data.frame(MEs0)

# Set rownames
rownames(MEs0) <- MEs0$sample_id
MEs0$sample_id <- NULL

# View
head(MEs0)

,MEpink,MEmagenta,MEorange,MElavenderblush3,MEmediumorchid,MEdarkturquoise,MEviolet,MEgrey60,MEantiquewhite4,MEplum2,⋯,MEbrown4,MEdarkorange,MEivory,MEorangered3,MElightgreen,MEmaroon,MEskyblue1,MEgreenyellow,MElightcyan1,MEgrey
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CO_R270X_B1_d18_A1,0.002823449,0.1258319,0.10469035,0.1311128,0.04044336,-0.0008011992,-0.03532585,0.13741148,0.1934694,0.1584736,⋯,-0.02140209,-0.1864026,-0.3010708,-0.237485289,-0.20880313,-0.1968823,-0.2377615,-0.07046394,-0.19793622,0.03688246
CO_R270X_B1_d18_A2,-0.005626526,0.1129932,0.08950843,0.2007168,0.10573560,-0.1026139157,-0.05867122,-0.03065719,0.1835488,0.1360300,⋯,-0.08840773,-0.1230010,-0.0300902,-0.071503428,-0.18307800,-0.3473836,-0.3290539,-0.05783421,-0.27733847,0.18298812
CO_R270X_B1_d18_A3,-0.009269397,0.1068721,0.06816713,0.1596962,0.06123052,-0.0779250899,-0.02830959,0.02831418,0.2258411,0.1489802,⋯,-0.09706673,-0.1175096,-0.1204957,-0.001908127,-0.14723315,-0.3185906,-0.3019938,-0.07210085,-0.21849367,0.09985123
CO_R270X_B1_d18_D1,0.056610933,0.1736676,0.15314122,0.2594875,0.13306242,0.0171832797,0.02338016,0.13518738,0.1904097,0.1861027,⋯,-0.15314491,-0.1315593,-0.1503681,-0.060190163,-0.05036689,-0.1951178,-0.2288203,-0.08289054,-0.09643528,0.05022305
CO_R270X_B1_d18_D2,0.010250562,0.1303987,0.12006893,0.2875670,0.10632790,-0.1236268993,-0.07346090,0.16142577,0.1583529,0.1225856,⋯,-0.17055588,-0.2044211,-0.2597181,-0.176214251,-0.09881005,-0.2801821,-0.2343696,-0.04828612,-0.10710889,0.14607728
CO_R270X_B1_d18_D3,-0.001516490,0.1256273,0.13218924,0.2724687,0.12930041,-0.1283232616,-0.11411616,0.13628788,0.1575974,0.1058645,⋯,-0.23491794,-0.2450841,-0.3306892,-0.210935083,-0.10745838,-0.2799510,-0.1854506,-0.07159546,-0.09121678,0.18240293


# Calculate Correlations

In [8]:
# Confirm sample order matches
table(rownames(MEs0) == rownames(metadata)) # All TRUE

# Pearson correlation
moduleTraitCorP <- WGCNA::cor(MEs0, metadata, method = "pearson", use = "p")
moduleTraitPvalueP <- corPvalueFisher(moduleTraitCorP, length(rownames(metadata)), twoSided = TRUE)

# Remove columns without significance
significant_modules <- rownames(moduleTraitPvalueP)[
  apply(moduleTraitPvalueP, 1, function(x) any(x < 0.05, na.rm = TRUE))
]

# Keep only modules with significant correlations
moduleTraitCorP <- moduleTraitCorP[significant_modules, ]
moduleTraitPvalueP <- moduleTraitPvalueP[significant_modules, ]

# Delete grey module
moduleTraitCorP <- moduleTraitCorP[!rownames(moduleTraitCorP) %in% "MEgrey", ]
moduleTraitPvalueP <- moduleTraitPvalueP[!rownames(moduleTraitPvalueP) %in% "MEgrey", ]

head(moduleTraitCorP)
head(moduleTraitPvalueP)


TRUE 
  32 

,D18_MUT_vs_D18_WT,D46_MUT_vs_D46_WT,D90_MUT_vs_D90_WT,Decreased_GAD1_Expression,Decreased_FOXG1_Expression,Smaller_Organoid_Area,Impaired_Neurite_Outgrowth
MEpink,-0.02897819,-0.1656571,0.4355235,0.110345317,0.07597858,0.4355235,0.4355235
MEorange,-0.30850572,0.1378302,-0.5902122,0.067603529,0.01244296,-0.5902122,-0.5902122
MElavenderblush3,-0.64159688,0.1639587,-0.4256317,0.083803333,-0.17596333,-0.4256317,-0.4256317
MEmediumorchid,-0.76963019,0.2049819,-0.5654669,0.005499157,-0.11000461,-0.5654669,-0.5654669
MEdarkturquoise,0.07046945,-0.4236572,0.1056415,-0.089328030,-0.02166447,0.1056415,0.1056415
MEviolet,0.21682919,-0.1751572,0.7208334,0.137735719,0.14737780,0.7208334,0.7208334


,D18_MUT_vs_D18_WT,D46_MUT_vs_D46_WT,D90_MUT_vs_D90_WT,Decreased_GAD1_Expression,Decreased_FOXG1_Expression,Smaller_Organoid_Area,Impaired_Neurite_Outgrowth
MEpink,8.759573e-01,0.36791377,1.196350e-02,0.5507362,0.6818447,1.196350e-02,1.196350e-02
MEorange,8.592585e-02,0.45506784,2.611229e-04,0.7154008,0.9465731,2.611229e-04,2.611229e-04
MElavenderblush3,4.176698e-05,0.37293637,1.437213e-02,0.6510127,0.3382860,1.437213e-02,1.437213e-02
MEmediumorchid,4.025153e-08,0.26283162,5.585246e-04,0.9763748,0.5519772,5.585246e-04,5.585246e-04
MEdarkturquoise,7.038566e-01,0.01489784,5.679811e-01,0.6295702,0.9071097,5.679811e-01,5.679811e-01
MEviolet,2.354272e-01,0.34055047,9.723749e-07,0.4553810,0.4240145,9.723749e-07,9.723749e-07


# Make Heatmap

In [9]:
# Represent significant P-values with * in heatmap
star <- apply(moduleTraitPvalueP, 2, function(x) {
  sapply(x, function(y) {
    ifelse(y < 0.001, "***", 
           ifelse(y < 0.01, "**", 
                  ifelse(y < 0.05, "*", "")))
  })
})

In [10]:
# Clean up column names
clean_colnames <- colnames(moduleTraitCorP) %>%
  gsub("_CtO_", " ", .) %>% 
  gsub("_", " ", .)

In [11]:
# Initialize PDF
pdf("08_WGCNA/Filtered_Module_Trait_Heatmap.pdf", width = 7, height = 10)

# Adjust the graphic window and margin settings
sizeGrWindow(width = 7, height = 10)
par(mar = c(8, 6, 2, 2))

# Plot the heatmap
labeledHeatmap(
  Matrix = moduleTraitCorP,
  xLabels = clean_colnames,
  yLabels = rownames(moduleTraitCorP),
  ySymbols = gsub("ME", "", rownames(moduleTraitCorP)),
  colorLabels = FALSE,
  colors = blueWhiteRed(50),
  textMatrix = star,
  setStdMargins = FALSE,
  cex.text = 0.4,
  textAdj = c(0.5, 0.8),
  zlim = c(-1, 1),
  main = "Module Trait Pearson Correlation",
  cex.lab.x = 0.5,
  cex.lab.y = 0.6,
  plotLegend = TRUE,
  legendLabel = "Pearson Correlation Coefficient",
  xLabelsAngle = 90
)

# Close the PDF device
dev.off()

pdf 
  2

In the above figure generate, blue is correlated with whatever we coded as 0 and red is correlated with whatever we coded as 1. Anything with a star means it's a significant correlation